# Guideline Impact Analysis

This notebook analyzes the impact of using different annotation guidelines on argument classification performance.

**Experimental Setup:**
- 4 datasets: ABSTRCT, ARGUMINSCI, PE, USELEC
- 6 guideline modes:
  1. **NO_GUIDELINE**: Simple 1-liner definition
  2. **CORRECT**: Each dataset uses its own matching guideline
  3. **ABSTRCT**: ABSTRCT guideline applied to all 4 datasets
  4. **ARGUMINSCI**: ARGUMINSCI guideline applied to all 4 datasets
  5. **PE**: PE guideline applied to all 4 datasets
  6. **USELEC**: USELEC guideline applied to all 4 datasets

**Research Question:** How much does using the correct dataset-specific guideline improve performance compared to using a generic definition or a mismatched guideline?

In [ ]:
import json
import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))
from config.paths import PROJECT_ROOT

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)

In [ ]:
# Load all experiment results
OUTPUT_DIR = PROJECT_ROOT / "experiments" / "zero_shot_guideline_outputs"
print(f"Loading from: {OUTPUT_DIR}")

GUIDELINE_MODES = ["NO_GUIDELINE", "CORRECT", "ABSTRCT", "ARGUMINSCI", "PE", "USELEC"]
DATASETS = ["ABSTRCT", "ARGUMINSCI", "PE", "USELEC"]

# Find and load the most recent file for each mode
results = {}
for mode in GUIDELINE_MODES:
    pattern = f"zero_shot_guideline_{mode}_*.json"
    files = sorted(OUTPUT_DIR.glob(pattern))
    if files:
        with open(files[-1]) as f:
            results[mode] = json.load(f)
        print(f"Loaded {mode}: {files[-1].name}")
    else:
        print(f"WARNING: No file found for {mode}")

print(f"\nLoaded {len(results)}/{len(GUIDELINE_MODES)} experiment files")

## 1. Overall Performance Comparison

In [ ]:
# Extract overall metrics for each mode
overall_metrics = []
for mode in GUIDELINE_MODES:
    if mode in results:
        metrics = results[mode]["summary"]["overall"]
        overall_metrics.append({
            "Guideline Mode": mode,
            "Accuracy": metrics["accuracy"],
            "Precision": metrics["precision"],
            "Recall": metrics["recall"],
            "F1 Score": metrics["f1_score"]
        })

df_overall = pd.DataFrame(overall_metrics)
print("Overall Performance by Guideline Mode:")
df_overall

In [ ]:
# Bar chart of overall accuracy by guideline mode
fig, ax = plt.subplots(figsize=(10, 6))

modes = df_overall["Guideline Mode"].tolist()
accs = df_overall["Accuracy"].tolist()

colors = []
for m in modes:
    if m == "NO_GUIDELINE":
        colors.append("gray")
    elif m == "CORRECT":
        colors.append("green")
    else:
        colors.append("steelblue")

bars = ax.bar(modes, accs, color=colors, edgecolor="black")

ax.set_ylabel("Accuracy", fontsize=12)
ax.set_xlabel("Guideline Mode", fontsize=12)
ax.set_title("Overall Accuracy by Guideline Mode", fontsize=14)
ax.set_ylim(0, 1)

for bar in bars:
    height = bar.get_height()
    ax.annotate(f"{height:.3f}",
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points",
                ha="center", va="bottom", fontsize=11)

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 2. Performance Matrix: Guideline Source vs Target Dataset (4x4)

This matrix shows how each guideline performs when applied to each dataset. The diagonal (when guideline matches dataset) should ideally show the best performance.

In [ ]:
# Build 4x4 matrix: rows = guideline used, columns = dataset evaluated on
matrix_data = []
for guideline in DATASETS:
    if guideline in results:
        row = {"Guideline": guideline}
        for dataset in DATASETS:
            metrics = results[guideline]["summary"]["by_dataset"][dataset]
            row[dataset] = metrics["accuracy"]
        matrix_data.append(row)

df_matrix = pd.DataFrame(matrix_data).set_index("Guideline")
print("Accuracy Matrix (rows: guideline used, columns: dataset evaluated)")
df_matrix

In [ ]:
# Heatmap visualization
fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(df_matrix, annot=True, fmt=".2f", cmap="RdYlGn", 
            vmin=0.4, vmax=1.0, ax=ax, linewidths=0.5,
            annot_kws={"size": 14})

# Highlight diagonal (correct guideline matches)
for i in range(len(DATASETS)):
    ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=False, edgecolor="black", lw=3))

ax.set_title("Accuracy: Guideline Source vs Target Dataset\n(diagonal = correct guideline)", fontsize=14)
ax.set_xlabel("Target Dataset", fontsize=12)
ax.set_ylabel("Guideline Source", fontsize=12)
plt.tight_layout()
plt.show()

## 3. Diagonal Analysis: Correct Guideline vs Mismatched

In [ ]:
# Compare diagonal (correct) vs off-diagonal (mismatched) performance
diagonal_scores = []
off_diagonal_scores = []

for i, guideline in enumerate(DATASETS):
    for j, dataset in enumerate(DATASETS):
        if guideline in df_matrix.index and dataset in df_matrix.columns:
            acc = df_matrix.loc[guideline, dataset]
            if i == j:  # Diagonal
                diagonal_scores.append({"Dataset": dataset, "Accuracy": acc})
            else:  # Off-diagonal
                off_diagonal_scores.append({
                    "Dataset": dataset, 
                    "Guideline": guideline,
                    "Accuracy": acc
                })

df_diagonal = pd.DataFrame(diagonal_scores)
df_off_diagonal = pd.DataFrame(off_diagonal_scores)

print("Correct Guideline Performance (diagonal):")
display(df_diagonal)

mean_correct = df_diagonal["Accuracy"].mean()
mean_mismatched = df_off_diagonal["Accuracy"].mean()

print(f"\nMean accuracy with CORRECT guideline: {mean_correct:.3f}")
print(f"Mean accuracy with MISMATCHED guideline: {mean_mismatched:.3f}")
print(f"Improvement from using correct guideline: +{mean_correct - mean_mismatched:.3f}")

## 4. Per-Dataset Analysis: Impact of Guideline Choice

In [ ]:
# For each dataset, compare: NO_GUIDELINE vs CORRECT vs all mismatched guidelines
comparison_data = []

for dataset in DATASETS:
    # NO_GUIDELINE performance
    if "NO_GUIDELINE" in results:
        no_guide_acc = results["NO_GUIDELINE"]["summary"]["by_dataset"][dataset]["accuracy"]
        comparison_data.append({"Dataset": dataset, "Mode": "NO_GUIDELINE", "Accuracy": no_guide_acc})
    
    # CORRECT guideline performance (from matrix diagonal)
    if dataset in df_matrix.index:
        correct_acc = df_matrix.loc[dataset, dataset]
        comparison_data.append({"Dataset": dataset, "Mode": "CORRECT", "Accuracy": correct_acc})
    
    # Each mismatched guideline
    for guideline in DATASETS:
        if guideline != dataset and guideline in df_matrix.index:
            mismatch_acc = df_matrix.loc[guideline, dataset]
            comparison_data.append({"Dataset": dataset, "Mode": f"{guideline}", "Accuracy": mismatch_acc})

df_comparison = pd.DataFrame(comparison_data)
df_comparison

In [ ]:
# Grouped bar chart: per dataset comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, dataset in enumerate(DATASETS):
    ax = axes[idx]
    subset = df_comparison[df_comparison["Dataset"] == dataset]
    
    modes = subset["Mode"].tolist()
    accuracies = subset["Accuracy"].tolist()
    
    colors = []
    for mode in modes:
        if mode == "NO_GUIDELINE":
            colors.append("gray")
        elif mode == "CORRECT":
            colors.append("green")
        else:
            colors.append("salmon")
    
    x_pos = range(len(modes))
    bars = ax.bar(x_pos, accuracies, color=colors, edgecolor="black")
    ax.set_xticks(x_pos)
    ax.set_xticklabels(modes, rotation=45, ha="right")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{dataset}", fontsize=12, fontweight="bold")
    ax.set_ylim(0, 1.1)
    
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f"{height:.2f}",
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=9)

plt.suptitle("Per-Dataset Performance by Guideline Mode\n(Gray=No Guideline, Green=Correct, Red=Mismatched)", 
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Summary Statistics

In [ ]:
# Calculate key statistics
no_guideline_acc = results["NO_GUIDELINE"]["summary"]["overall"]["accuracy"] if "NO_GUIDELINE" in results else None
correct_acc = results["CORRECT"]["summary"]["overall"]["accuracy"] if "CORRECT" in results else None

# Average of mismatched guidelines (off-diagonal)
mismatched_accs = []
for guideline in DATASETS:
    if guideline in results:
        for dataset in DATASETS:
            if guideline != dataset:
                mismatched_accs.append(results[guideline]["summary"]["by_dataset"][dataset]["accuracy"])

avg_mismatched = np.mean(mismatched_accs) if mismatched_accs else None

print("=" * 60)
print("SUMMARY: Guideline Impact Analysis")
print("=" * 60)
if no_guideline_acc is not None:
    print(f"\n1. NO_GUIDELINE (simple 1-liner):       {no_guideline_acc:.3f}")
if correct_acc is not None:
    print(f"2. CORRECT guideline (diagonal):        {correct_acc:.3f}")
if avg_mismatched is not None:
    print(f"3. MISMATCHED guideline (off-diag avg): {avg_mismatched:.3f}")

print("\n" + "-" * 60)
if no_guideline_acc and correct_acc:
    diff = correct_acc - no_guideline_acc
    pct = (diff / no_guideline_acc) * 100
    print(f"Improvement: NO_GUIDELINE -> CORRECT:   {diff:+.3f} ({pct:+.1f}%)")
if avg_mismatched and correct_acc:
    diff = correct_acc - avg_mismatched
    pct = (diff / avg_mismatched) * 100
    print(f"Improvement: MISMATCHED -> CORRECT:     {diff:+.3f} ({pct:+.1f}%)")
if no_guideline_acc and avg_mismatched:
    diff = no_guideline_acc - avg_mismatched
    print(f"Difference:  NO_GUIDELINE vs MISMATCHED: {diff:+.3f}")
print("=" * 60)

In [ ]:
# Final comparison bar chart
fig, ax = plt.subplots(figsize=(8, 6))

categories = ["NO_GUIDELINE\n(1-liner)", "MISMATCHED\n(wrong guideline)", "CORRECT\n(matching guideline)"]
values = [no_guideline_acc or 0, avg_mismatched or 0, correct_acc or 0]
colors = ["gray", "salmon", "green"]

bars = ax.bar(categories, values, color=colors, edgecolor="black", width=0.6)

ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Impact of Guideline Specificity on Classification Accuracy", fontsize=14)
ax.set_ylim(0, 1)

for bar in bars:
    height = bar.get_height()
    if height > 0:
        ax.annotate(f"{height:.3f}",
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

## 6. F1 Score Analysis

In [ ]:
# Build F1 matrix
f1_matrix_data = []
for guideline in DATASETS:
    if guideline in results:
        row = {"Guideline": guideline}
        for dataset in DATASETS:
            metrics = results[guideline]["summary"]["by_dataset"][dataset]
            row[dataset] = metrics["f1_score"]
        f1_matrix_data.append(row)

df_f1_matrix = pd.DataFrame(f1_matrix_data).set_index("Guideline")
print("F1 Score Matrix (rows: guideline used, columns: dataset evaluated)")
df_f1_matrix

In [ ]:
# F1 Heatmap
fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(df_f1_matrix, annot=True, fmt=".2f", cmap="RdYlGn", 
            vmin=0.4, vmax=1.0, ax=ax, linewidths=0.5,
            annot_kws={"size": 14})

# Highlight diagonal
for i in range(len(DATASETS)):
    ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=False, edgecolor="black", lw=3))

ax.set_title("F1 Score: Guideline Source vs Target Dataset\n(diagonal = correct guideline)", fontsize=14)
ax.set_xlabel("Target Dataset", fontsize=12)
ax.set_ylabel("Guideline Source", fontsize=12)
plt.tight_layout()
plt.show()

## 7. Key Findings

In [ ]:
# Identify best and worst cross-guideline performance
cross_results = []
for guideline in DATASETS:
    if guideline not in df_matrix.index:
        continue
    for dataset in DATASETS:
        if guideline != dataset and dataset in df_matrix.columns:
            acc = df_matrix.loc[guideline, dataset]
            correct_acc_val = df_matrix.loc[dataset, dataset]
            cross_results.append({
                "Guideline": guideline,
                "Dataset": dataset,
                "Accuracy": acc,
                "Correct Acc": correct_acc_val,
                "Drop": correct_acc_val - acc
            })

df_cross = pd.DataFrame(cross_results)

print("Largest Performance Drops (when using wrong guideline):")
display(df_cross.nlargest(5, "Drop"))

print("\nSmallest Performance Drops (guidelines that transfer well):")
display(df_cross.nsmallest(5, "Drop"))